# Infaction — Political dominance 5pp experiment (Zone + Site + Location)
Uses v0.4.8 Site design, historic v0.4.6 upstream generator, and a SEPARATE experimental political-ownership overlay. Existing locked geographic placement is not changed.
After the original six-Faction + Neutral two-way 50:50 A/B calculation across all generated Zone + Site + Location Places, each Zone/Location explicitly selected by a Faction gets 95% of its initial 100-point vector plus +5 percentage points for the selecting Faction. Sites get no selector bonus. Then classify Single/Dual/none. JSON contains political_dominance and Place node annotations. The Land and City remain outside this ownership experiment.
**Data gap:** eight original Scale-3 Place names lack Type/Base affinity and Static Neutral source data. If generated as Sites, the overlay reports partial_missing_source_affinity with their names. Site thresholds provisionally reuse Location thresholds; this is NOT locked. Upstream generation remains historical v0.4.6.


In [ ]:
import urllib.request,json,contextlib,io,types
ROOT="https://raw.githubusercontent.com/playrenathea/rantau/main/"
def get_text(path):return urllib.request.urlopen(ROOT+path,timeout=60).read().decode("utf-8")
base=json.loads(get_text("notebooks/Infaction_Locked_City_Placement_v0.4.6.ipynb"))
scope={"__name__":"__main__"}
for idx in (1,2,3):
    exec(compile("".join(base["cells"][idx]["source"]),f"base_cell_{idx}","exec"),scope)
print("Canonical old executable selection/placement/City preflight passed")


In [ ]:
from pathlib import Path
site_mod=types.ModuleType("infaction_site_proximity_checkpoint")
exec(compile(get_text("infaction_site_proximity_checkpoint.py"),"infaction_site_proximity_checkpoint.py","exec"),site_mod.__dict__)
master=scope["MASTER"]
scales={name:int(row["scale"]) for name,row in master["raw"]["place_catalog"].items()}
effective={**scales,**{"Fortress":3,"Palace":3,"Mine":3,"Factory":3,"Town Hall":2}}
site_candidates=[p for p in scope["place_names"] if effective.get(p) in (2,3,4)]
assert len(site_candidates)==50
site_city_scores={p:int(row["inside_city_original"]) for p,row in master["raw"]["place_catalog"].items()}
assert set(site_candidates).issubset(site_city_scores)
site_result=site_mod.group_sites(scope["placed"],scope["result"],scope["SITE_AFFINITY"],site_candidates,site_city_scores,seed=scope["SEED"])
assert site_result["grouped_location_count"]+site_result["ungrouped_location_count"]==len(scope["placed"]["locations"])
family_mod=types.ModuleType("infaction_targeted_farm_ship_site_prune_v2")
exec(compile(get_text("experiments/infaction_targeted_farm_ship_site_prune_v2.py"),"infaction_targeted_farm_ship_site_prune_v2.py","exec"),family_mod.__dict__)
family_rules=json.loads(get_text("experiments/infaction_targeted_farm_ship_site_filter_v2.json"))
site_records=[{"site":s["name"],"container":("z:"+s["parent_zone"] if s["parent_zone"] is not None else s["city"].lower()),"children":s["children"]} for s in site_result["sites"]]
family_result=family_mod.prune_sites(site_records,family_rules,proximity=scope["SITE_AFFINITY"])
sites_by_name={s["name"]:s for s in site_result["sites"]}
retained_names={r["site"] for r in family_result["sites"]}
retained_sites=[{**sites_by_name[r["site"]],"children":r["children"]} for r in family_result["sites"]]
remaining_by_zone={z:list(members) for z,members in site_result["direct_locations_by_zone"].items()}
remaining_inside=list(site_result["standalone_ungrouped_inside"])
remaining_outside=list(site_result["standalone_ungrouped_outside"])
for container,children in family_result["direct_restored"].items():
    if container.startswith("z:"):
        remaining_by_zone[container[2:]].extend(children)
    elif container=="inside":
        remaining_inside.extend(children)
    elif container=="outside":
        remaining_outside.extend(children)
    else:
        raise AssertionError("Unknown restored container")
members=[loc for s in retained_sites for loc in s["children"]]
members+=sum(remaining_by_zone.values(),[])+remaining_inside+remaining_outside
assert len(members)==len(set(members))==len(scope["placed"]["locations"])
assert set(members)==set(scope["placed"]["locations"])
site_result["sites"]=retained_sites
site_result["direct_locations_by_zone"]=remaining_by_zone
site_result["standalone_ungrouped_inside"]=remaining_inside
site_result["standalone_ungrouped_outside"]=remaining_outside
site_result["grouped_location_count"]=sum(len(s["children"]) for s in retained_sites)
site_result["ungrouped_location_count"]=len(members)-site_result["grouped_location_count"]
site_result["family_filter"]={k:v for k,v in family_result.items() if k!="sites"}
site_result["status"]="Proximity with Farm/Ship family post-filter v2; historical selection executable"
# Post-Site rules operate after family prune, without rerunning the original greedy worlds.
post_mod=types.ModuleType("infaction_site_postprocess_v3")
exec(compile(get_text("infaction_site_postprocess_v3.py"),"infaction_site_postprocess_v3.py","exec"),post_mod.__dict__)
# The direct pools above already contain family-restored Locations: don't restore twice.
family_after={**family_result,"direct_restored":{}}
direct_pools={"z:"+z:list(ls) for z,ls in remaining_by_zone.items()}
direct_pools["inside"]=list(remaining_inside)
direct_pools["outside"]=list(remaining_outside)
post=post_mod.apply_garden_and_replacements(
    site_records,family_after,direct_pools,
    scope["placed"]["zones"],scope["placed"]["locations"],site_candidates,
    scope["SITE_AFFINITY"],site_city_scores,family_rules,scope["SEED"])
def to_site_record(record):
    container=record["container"]
    zone=container[2:] if container.startswith("z:") else None
    city_label=scope["result"]["zones"][zone]["city"] if zone is not None else container.title()
    oldname="Garden" if record["site"]=="Grounds" else record["site"]
    original=sites_by_name.get(oldname,{})
    return {"name":record["site"],"parent_zone":zone,"city":city_label,
            "children":record["children"],"scores":original.get("scores",{
                "gap":record.get("gap"),"h2":record.get("h2"),"lowest":record.get("low")}),
            **({"replaces":record["replaces"]} if "replaces" in record else {})}
site_result["sites"]=[to_site_record(s) for s in post["sites"]]
site_result["direct_locations_by_zone"]={z:post["direct_pools"]["z:"+z] for z in scope["placed"]["zones"]}
site_result["standalone_ungrouped_inside"]=post["direct_pools"]["inside"]
site_result["standalone_ungrouped_outside"]=post["direct_pools"]["outside"]
site_result["grouped_location_count"]=sum(len(s["children"]) for s in site_result["sites"])
site_result["ungrouped_location_count"]=len(scope["placed"]["locations"])-site_result["grouped_location_count"]
site_result["postprocessing"]={
    "version":post["version"],"garden_splits":post["garden_splits"],
    "replacement_sites":post["replacements"],"unreplaced":post["unreplaced"]}
site_result["status"]="Book1 Proximity + Farm/Ship family v2 + Garden/Grounds split + optional replacement"
assert site_result["grouped_location_count"]+site_result["ungrouped_location_count"]==len(scope["placed"]["locations"])
print("Seed",scope["SEED"],"Sites",len(site_result["sites"]),
      "Garden split",len(post["garden_splits"]),
      "replacements",len(post["replacements"]),"unreplaced",len(post["unreplaced"]))
for s in site_result["sites"]:
    print(s["name"],"in",s["parent_zone"] or "Standalone",":",", ".join(s["children"]))
Path(f'infaction_world_with_sites_experimental_{scope["SEED"]}.json').write_text(
    json.dumps({"world":scope["world"],"placement":scope["placed"],
                "city":scope["result"],"site_grouping":site_result},
               indent=2,ensure_ascii=False))


In [ ]:
# Export a complete, navigable world Place tree and full Faction attributes.
export_mod=types.ModuleType('infaction_world_json_export_v048')
exec(compile(get_text('infaction_world_json_export_v048.py'),'infaction_world_json_export_v048.py','exec'),export_mod.__dict__)
full_world=export_mod.build_world_export(
    scope['world'], scope['placed'], scope['result'], site_result,
    place_catalog=scope['MASTER']['raw']['place_catalog'])
assert len(full_world['factions'])==6
assert len(full_world['places']['nodes']) == 2+len(scope['placed']['zones'])+len(scope['placed']['locations'])+len(site_result['sites'])
assert all('base_affinities' in f and 'animal_affinities' in f and 'types' in f for f in full_world['factions'])
# Experimental political dominance; do not mutate original geographic placement.
pol_mod=types.ModuleType('infaction_political_dominance_5pp_v1')
exec(compile(get_text('experiments/infaction_political_dominance_5pp_v1.py'),
             'infaction_political_dominance_5pp_v1.py','exec'),pol_mod.__dict__)
political=pol_mod.calculate(
    scope['world'],site_result,
    scope['ZONE_DATA']['zones'],scope['LOCATION_DATA']['locations'],
    scope['neutral'])
full_world['political_dominance']=political
for place_name,detail in political['places'].items():
    role=detail['role']
    key=('zone:' if role=='zone' else 'site:' if role=='site' else 'location:')+place_name
    full_world['places']['nodes'][key]['attributes']['political_dominance']=detail
full_world['status']['political_dominance']='experimental; not locked'
if political['missing_source_places']:
    print('WARNING: Missing source political affinity:',political['missing_source_places'])
else:
    print('PASS: all Zone/Site/Location ownership scores source-backed')
print('Political ownership counts:',political['counts'])
export_path=Path(f"infaction_political5pp_world_seed_{scope['SEED']}_v0.4.8-site.json")
export_mod.save_world_json(full_world,export_path)
print('PASS JSON:',export_path,'size',export_path.stat().st_size,'bytes')
print('Faction count:',len(full_world['factions']))
print('Place counts:',full_world['places']['counts'])
print('Root children:',[full_world['places']['nodes'][i]['name'] for i in full_world['places']['nodes']['land']['children_ids']])
print('Runtime status:',full_world['status']['upstream_runtime'])
# Download the saved JSON from Colab; this does not affect model or sources.
try:
    from google.colab import files
    files.download(str(export_path))
except ImportError:
    print('Saved locally:',export_path.resolve())
